In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import datetime
import scipy.stats
from scipy.optimize import curve_fit
#import plotly.offline as pyo
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.colors as cl
import matplotlib.dates as mdates
import matplotlib
import matplotlib.colors as mcolors
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
import seaborn as sns
import netCDF4 as nc
import xarray as xr
#import cartopy.crs as ccrs
#from cartopy import feature as cfa
import os
import plotly.io as pio
pio.renderers.default = 'notebook'
from sklearn.metrics import r2_score

In [ ]:
def decimal_year_to_datetime(decimal_year):
    year = decimal_year.astype(int)
    decimal_part = decimal_year - year
    start_of_year = pd.to_datetime([f'{y}-01-01' for y in year])
    seconds_in_year = (pd.to_datetime([f'{y+1}-01-01' for y in year]) - start_of_year).total_seconds()
    seconds = decimal_part * seconds_in_year
    return start_of_year + pd.to_timedelta(seconds, unit='s')

def datetime_to_decimal_year(datetime_):
    year = datetime_.year
    days = (datetime_.date() - datetime.date(year, 1,1)).days
    res = year + days/365.25
    return res.dt.date


def FFT_res(x, si):
    n = len(x)
    x_FFT_shifted = np.fft.fftshift(np.fft.fft(x))
    x_FFT_freq_shifted = np.fft.fftshift(np.fft.fftfreq(n, d=si))
    return x_FFT_freq_shifted, x_FFT_shifted


def fft_noramlize(fft_res):
    n = len(fft_res)
    norm_fft = np.abs(fft_res)/(np.max(np.abs(fft_res)))
    return norm_fft



In [ ]:
%run ../20260223_read_data.py
df_resampled

In [ ]:
data_old = pd.read_excel('../../../2025-02-10_Depth/20251126_DepthLaurenPaper/CoralDepthData.xlsx')
data_old.rename(columns={data_old.columns[0]:'id', data_old.columns[1]:'d13c', data_old.columns[2]:'d18o'}, inplace=True)
data_old['path'] = [x[1] for x in data_old['id'].str.split(' ')]
data_old['idx'] = [x[-1][0] for x in data_old['id'].str.split(' ')]
depths_old = np.array([-105, -95, -68, -58, -48, -35, -15]) - 60
#depths = np.array([-105, -95, -73, -63, -53, -42, -32.5]) - 60
paths_old = np.unique(data_old['path'].values)
path_depth_old = dict(zip(paths_old, depths_old))
data_old['depth'] = data_old['path'].map(path_depth_old)
data_old

In [ ]:
data = pd.read_csv('memo_age_shifted_fred_v1.csv')
data['age_shifted'] = pd.to_numeric(data['age_shifted'])
data['age_shifted_dt'] = decimal_year_to_datetime(data['age_shifted'])
data.set_index('age_shifted_dt', inplace=True)
paths = np.sort(data['dataset'].unique())
depths = np.sort(data['depth'].unique())


freq = 'ME'   # 2-month end-of-month; change to 'ME' for monthly, 'QE' for quarterly

# Build the shared time index spanning all datasets
anchor = pd.Timestamp("1995-01-31")

full_idx = pd.date_range(
    start=anchor,
    end='01-31-1997',
    freq=freq
)

resampled_parts = []

for p in paths:
    data_iter = data[data['dataset']==p].copy()
    data_iter = data_iter[~data_iter.index.duplicated()]
    data_iter = data_iter.sort_index()
    
    data_res = data_iter[['age_shifted','shift','d18o','d13c']].resample(freq).mean()
    data_res = data_res.reindex(full_idx)
    data_res = data_res.interpolate(method='time', limit=2, limit_direction='both')
    
    data_res.columns = [f'{p}_{c}' for c in data_res.columns]
    data_res[p+'_depth'] = data_iter['depth'].unique()[0]
  
    resampled_parts.append(data_res)

df_resampled_depth = pd.concat(resampled_parts, axis=1).resample('2ME').mean()

df_resampled_depth = df_resampled_depth.join(df_resampled[['d18o','d13c','age_final']]).rename(columns={'d18o':'H_d18o','d13c':'H_d13c','age_final':'H_age_final'})
df_resampled_depth['H_depth'] = -35-60
paths = np.append(paths, 'H')
depths = np.append(depths, -95)
df_resampled_depth


data = pd.read_csv('memo_age_shifted_fred.csv')
data['age_shifted'] = pd.to_numeric(data['age_shifted'])
data['age_shifted_dt'] = decimal_year_to_datetime(data['age_shifted'])
data.set_index('age_shifted_dt', inplace=True)
paths = data['dataset'].unique()
depth = data['depth'].unique()


freq = 'ME'   # 2-month end-of-month; change to 'ME' for monthly, 'QE' for quarterly

# Build the shared time index spanning all datasets
anchor = pd.Timestamp("1995-01-31")

full_idx = pd.date_range(
    start=anchor,
    end='01-31-1997',
    freq=freq
)

resampled_parts = []

for p in paths:
    data_iter = data[data['dataset']==p].copy()
    data_iter = data_iter[~data_iter.index.duplicated()]
    data_iter = data_iter.sort_index()
    
    data_res = data_iter[['age_shifted','shift','d18o','d13c']].resample(freq).mean()
    data_res = data_res.reindex(full_idx)
    data_res = data_res.interpolate(method='time', limit=2, limit_direction='both')
    
    data_res.columns = [f'{p}_{c}' for c in data_res.columns]
    data_res['depth'] = data_iter['depth'].unique()[0]
  
    resampled_parts.append(data_res)

df_resampled = pd.concat(resampled_parts, axis=1).resample('2ME').mean()
df_resampled

In [ ]:
data_iter

In [ ]:
paths

In [ ]:
colors = ['#F58727', '#F5EE27', '#95F527', '#2795F5', '#8727F5', '#540505', '#000000']

fig = make_subplots(specs=[[{"secondary_y": True}]])

for i,p in enumerate(paths):

    data_iter = data[data['dataset']==p]
    fig.add_trace(go.Scatter(x=data_iter['age_shifted'], y=data_iter['d18o'] , mode="lines+markers", line_color= colors[i], text = data_iter['point_idx'] , name=p ), secondary_y=False)

#df_sorted_selected = df_sorted[df_sorted['age_final']]
fig.add_trace(go.Scatter(x=df_resampled_depth['H_age_final'], y=df_resampled_depth['H_d18o'] , mode="lines+markers", line_color='#000000', text = 'H' , name='H' ), secondary_y=False)

#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=True)



#fig.update_layout(
#    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)
#fig.update_layout(
#    xaxis2=dict(title="Secondary X-axis Title", overlaying="x", side="top", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.update_layout(width=1200, height=500, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{18}$ O Observerd', secondary_y=False)
fig.update_yaxes(title_text= r'$\delta^{18}$ O Synthetic', secondary_y=True)

fig.show()
#fig.write_html('d18o_syntetic-d18O.html')


In [ ]:
df_resampled_depth

In [ ]:
colors = ['#F58727', '#F5EE27', '#95F527', '#2795F5', '#8727F5', '#540505', '#000000']

fig = make_subplots(specs=[[{"secondary_y": True}]])

for i,p in enumerate(paths):

    data_iter = data[data['dataset']==p]
    fig.add_trace(go.Scatter(x=df_resampled_depth.index, y=df_resampled_depth[p+'_d18o'] , mode="lines", line_color= colors[i], text = data_iter['point_idx'] , name=p ), secondary_y=False)

#df_sorted_selected = df_sorted[df_sorted['age_final']]
#fig.add_trace(go.Scatter(x=df_sorted_selected['age_dt'], y=df_sorted_selected['d18o'] , mode="lines", line_color='#000000', text = 'H' , name='H' ), secondary_y=False)

#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=True)



#fig.update_layout(
#    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)
#fig.update_layout(
#    xaxis2=dict(title="Secondary X-axis Title", overlaying="x", side="top", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.update_layout(width=1200, height=500, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{18}$ O Observerd', secondary_y=False)
fig.update_yaxes(title_text= r'$\delta^{18}$ O Synthetic', secondary_y=True)

fig.show()
#fig.write_html('d18o_syntetic-d18O.html')


In [ ]:
colors = ['#F58727', '#F5EE27', '#95F527', '#2795F5', '#8727F5', '#540505', '#000000']

fig = make_subplots(specs=[[{"secondary_y": True}]])

for i,p in enumerate(paths):

    data_iter = data[data['dataset']==p]
    fig.add_trace(go.Scatter(x=df_resampled_depth.index, y=df_resampled_depth[p+'_d13c'] , mode="markers", line_color= colors[i], text = data_iter['point_idx'] , name=p ), secondary_y=False)

#df_sorted_selected = df_sorted[df_sorted['age_final']]
#fig.add_trace(go.Scatter(x=df_sorted_selected['age_dt'], y=df_sorted_selected['d18o'] , mode="lines", line_color='#000000', text = 'H' , name='H' ), secondary_y=False)

#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=True)



#fig.update_layout(
#    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)
#fig.update_layout(
#    xaxis2=dict(title="Secondary X-axis Title", overlaying="x", side="top", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.update_layout(width=1200, height=500, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{13}$ C Observerd', secondary_y=False)
fig.update_yaxes(title_text= r'$\delta^{13}$ C Synthetic', secondary_y=True)

fig.show()
#fig.write_html('d18o_syntetic-d18O.html')


In [ ]:
df_resampled_depth

In [ ]:
def line_fit(X,Y):
    n = len(X)
    P, residuals, rank, singular_values, rcond= np.polyfit(X,Y,1, full=True)
    dx = np.max(X) - np.min(X)
    X_new = np.array([np.min(X)-dx*0.2, np.max(X)+dx*0.2])
    Y_new = np.polyval(P, X_new)
    rmse = np.sqrt(residuals[0] / n)
    #Create a polynomial function from the coefficients
    pol = np.poly1d(P)
    Y_Pred = pol(X)
    r_squared = r2_score(Y, Y_Pred)
    return X_new, Y_new, P, rmse, r_squared

In [ ]:
df_resampled_depth.loc[:, df_resampled_depth.columns.str.endswith('_depth')].mean()

In [ ]:
df_resampled_depth.loc[:, df_resampled_depth.columns.str.endswith('_d18o')].mean()

In [ ]:
colors = ['#F58727', '#F5EE27', '#95F527', '#2795F5', '#8727F5', '#540505', '#000000']
colormap_viri = matplotlib.colormaps.get_cmap('viridis') 
colormap_inf = matplotlib.colormaps.get_cmap('inferno_r') 
normalize_viri = mcolors.Normalize(vmin=np.min(depths), vmax=np.max(depths))
normalize_inf = mcolors.Normalize(vmin=np.min(depths_old), vmax=np.max(depths_old))


fig, ax = plt.subplots(1, figsize=(10, 6))

#ax.scatter(df_resampled['A_d13c'], np.repeat(depth[0], 25 ))

for p,d,i in zip(paths, depths, range(len(depths))):

    #ax.scatter(df_resampled_depth[p+'_d13c'], df_resampled_depth[p+'_depth'], c=colors[i] ,label=p)
    ax.scatter(df_resampled_depth[p+'_d13c'].median(), df_resampled_depth[p+'_depth'].median(),
               s=200 ,c= df_resampled_depth[p+'_depth'].median(), 
               cmap=colormap_viri , 
               norm=normalize_viri, marker='^')
    ax.text(df_resampled_depth[p+'_d13c'].median(), df_resampled_depth[p+'_depth'].median()+5, 
            p, color='red', horizontalalignment='center', verticalalignment='center')
    

for p,d,i in zip(paths_old, depths_old, range(len(depths_old))):

    data_iter = data_old[data_old['path']==p]
    #ax.scatter(data_iter['d13c'], data_iter['depth'], c=colors[i] ,label=p)
    ax.scatter(data_iter['d13c'].median(), data_iter['depth'].median(), s=200 ,c= data_iter['depth'].median(), cmap=colormap_inf , norm=normalize_inf, marker='v')
    ax.text(data_iter['d13c'].median(), data_iter['depth'].median()-5, 
            p, color='green', horizontalalignment='center', verticalalignment='center')
    



x_ = df_resampled_depth.loc[:, df_resampled_depth.columns.str.endswith('_d13c')].median()
y_ = df_resampled_depth.loc[:, df_resampled_depth.columns.str.endswith('_depth')].mean()
X_, Y_, P_, rmse_, r2_= line_fit(x_,y_)

#ax.plot(X_,Y_, color='g', label=f'y = {P_[0]:.3f}x+{P_[1]:.3f}'+'\n'+
#        f'RMSE={rmse_:.3f}'+'\n'+
#       f'$R^2$={r2_:.3f}')

x = data_old.groupby('path')['d13c'].median().values
y = depths_old
X, Y, P, rmse, r2= line_fit(x,y)

#ax.plot(X,Y, color='r', label=f'y = {P[0]:.3f}x+{P[1]:.3f}'+'\n'+
#        f'RMSE={rmse:.3f}'+'\n'+
#       f'$R^2$={r2:.3f}')


x_merged = np.append(x_, x)
y_merged = np.append(y_, y)


X_merged, Y_merged, P_merged, rmse_merged, r2_merged= line_fit(x_merged, y_merged)

#ax.plot(X_merged, Y_merged, color='b', label=f'y = {P_merged[0]:.3f}x+{P_merged[1]:.3f}'+'\n'+
#        f'RMSE={rmse_merged:.3f}'+'\n'+
#       f'$R^2$={r2_merged:.3f}')


ax.legend()
ax.grid()

#fig.savefig('depth_d13c.pdf')

In [ ]:
colors = ['#F58727', '#F5EE27', '#95F527', '#2795F5', '#8727F5', '#540505', '#000000']
colormap_viri = matplotlib.colormaps.get_cmap('viridis') 
colormap_inf = matplotlib.colormaps.get_cmap('inferno_r') 
normalize_viri = mcolors.Normalize(vmin=np.min(depths), vmax=np.max(depths))
normalize_inf = mcolors.Normalize(vmin=np.min(depths_old), vmax=np.max(depths_old))


fig, ax = plt.subplots(1, figsize=(10, 6))

#ax.scatter(df_resampled['A_d13c'], np.repeat(depth[0], 25 ))

for p,d,i in zip(paths, depths, range(len(depths))):

    #ax.scatter(df_resampled_depth[p+'_d13c'], df_resampled_depth[p+'_depth'], c=colors[i] ,label=p)
    ax.scatter(df_resampled_depth[p+'_d13c'].mean(), df_resampled_depth[p+'_depth'].mean(),
               s=200 ,c= df_resampled_depth[p+'_depth'].mean(), 
               cmap=colormap_viri , 
               norm=normalize_viri, marker='^')
    ax.text(df_resampled_depth[p+'_d13c'].mean(), df_resampled_depth[p+'_depth'].mean()+5, 
            p, color='red', horizontalalignment='center', verticalalignment='center')
    

for p,d,i in zip(paths_old, depths_old, range(len(depths_old))):

    data_iter = data_old[data_old['path']==p]
    #ax.scatter(data_iter['d13c'], data_iter['depth'], c=colors[i] ,label=p)
    ax.scatter(data_iter['d13c'].mean(), data_iter['depth'].mean(), s=200 ,c= data_iter['depth'].mean(), cmap=colormap_inf , norm=normalize_inf, marker='v')
    ax.text(data_iter['d13c'].mean(), data_iter['depth'].mean()-5, 
            p, color='green', horizontalalignment='center', verticalalignment='center')
    



x_ = df_resampled_depth.loc[:, df_resampled_depth.columns.str.endswith('_d13c')].mean()
y_ = df_resampled_depth.loc[:, df_resampled_depth.columns.str.endswith('_depth')].mean()

X_, Y_, P_, rmse_, r2_= line_fit(x_,y_)

ax.plot(X_, Y_, color='g', label=f'y = {P_[0]:.3f}x+{P_[1]:.3f}'+'\n'+
        f'RMSE={rmse_:.3f}'+'\n'+
       f'$R^2$={r2_:.3f}')

x = data_old.groupby('path')['d13c'].mean().values
y = depths_old
X, Y, P, rmse, r2= line_fit(x,y)
ax.plot(X,Y, color='r', label=f'y = {P[0]:.3f}x+{P[1]:.3f}'+'\n'+
        f'RMSE={rmse:.3f}'+'\n'+
       f'$R^2$={r2:.3f}')


x_merged = np.append(x_, x)
y_merged = np.append(y_, y)


X_merged, Y_merged, P_merged, rmse_merged, r2_merged= line_fit(x_merged, y_merged)

ax.plot(X_merged, Y_merged, color='b', label=f'y = {P_merged[0]:.3f}x+{P_merged[1]:.3f}'+'\n'+
        f'RMSE={rmse_merged:.3f}'+'\n'+
       f'$R^2$={r2_merged:.3f}')


ax.legend()
ax.grid()

fig.savefig('depth_d13c.pdf')

In [ ]:
colors = ['#F58727', '#F5EE27', '#95F527', '#2795F5', '#8727F5', '#540505', '#000000']


fig, ax = plt.subplots(1, figsize=(6, 6))

#ax.scatter(df_resampled['A_d13c'], np.repeat(depth[0], 25 ))

for p,d,i in zip(paths, depths, range(len(depths))):

    ax.scatter(df_resampled_depth[p+'_d13c'], df_resampled_depth[p+'_depth'], c=colors[i] ,label=p)
    ax.scatter(df_resampled_depth[p+'_d13c'].mean(), df_resampled_depth[p+'_depth'].mean(), s=200 ,c=colors[i] ,  marker='^')

#X, Y, P, rmse, r2= line_fit(x,y)
for p,d,i in zip(paths_old, depths_old, range(len(depths_old))):

    data_iter = data_old[data_old['path']==p]
    ax.scatter(data_iter['d13c'], data_iter['depth'], c=colors[i] ,label=p)
    ax.scatter(data_iter['d13c'].mean(), data_iter['depth'].mean(), s=200 ,c=colors[i] ,  marker='^')


ax.legend()

In [ ]:
data_old

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.lines import Line2D

# ─────────────────────────────────────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────────────────────────────────────
def decimal_year_to_datetime(decimal_years):
    decimal_years = pd.to_numeric(decimal_years, errors='coerce')
    year = decimal_years.astype(int)
    remainder = decimal_years - year
    days_in_year = year.map(
        lambda y: 366 if (y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)) else 365
    )
    day_of_year = (remainder * days_in_year).astype(int)
    base = pd.to_datetime(year.astype(str) + '-01-01')
    return base + pd.to_timedelta(day_of_year, unit='D')

df = pd.read_csv('memo_age_shifted_fred_v1.csv')
df['dt'] = decimal_year_to_datetime(df['age_shifted'])
df = df.set_index('dt')

# Dataset metadata: depth, colour, marker
DS_META = {
    'A': {'depth': -160.0,  'color': '#a870e0', 'marker': 'o', 'label': 'A  −160 m'},
    'B': {'depth': -154.0,  'color': '#60a0e0', 'marker': 'o', 'label': 'B  −154 m'},
    'F': {'depth': -145.0,  'color': '#ffd700', 'marker': 'D', 'label': 'F  −145 m (FRED)'},
    'C': {'depth': -130.0,  'color': '#70c090', 'marker': 'o', 'label': 'C  −130 m'},
    'D': {'depth': -118.5,  'color': '#f0c060', 'marker': 'o', 'label': 'D  −118.5 m'},
    'E': {'depth': -108.0,  'color': '#ff8060', 'marker': 'o', 'label': 'E  −108 m'},
}
DEPTHS_ORDER = ['A', 'B', 'F', 'C', 'D', 'E']   # top→bottom on Y axis

# ─────────────────────────────────────────────────────────────────────────────
# RESAMPLE TO SHARED BI-MONTHLY GRID
# ─────────────────────────────────────────────────────────────────────────────
freq = 'ME'   # monthly then thin — avoids misaligned 2ME bins
full_idx = pd.date_range(start='1994-08-31', end='1999-04-30', freq=freq)

resampled = {}
for ds in DEPTHS_ORDER:
    sub = df[df['dataset'] == ds][['d13c', 'd18o']].copy()
    sub = sub[~sub.index.duplicated()].sort_index()
    res = sub.resample(freq).mean().reindex(full_idx)
    res = res.interpolate(method='time', limit=3, limit_direction='both')
    resampled[ds] = res

# Thin to every 2 months
timestep_idx = full_idx[::2]
N = len(timestep_idx)

# ─────────────────────────────────────────────────────────────────────────────
# COLOUR RAMP FOR TIME STEPS  (teal → amber)
# ─────────────────────────────────────────────────────────────────────────────
cmap = plt.cm.coolwarm_r
ts_colors = [cmap(i / (N - 1)) for i in range(N)]

# ─────────────────────────────────────────────────────────────────────────────
# PLOT
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 8))
fig.patch.set_facecolor('#0e1117')
ax.set_facecolor('#0e1117')

depth_positions = {ds: DS_META[ds]['depth'] for ds in DEPTHS_ORDER}
depths_y = list(depth_positions.values())

for i, ts in enumerate(timestep_idx):
    col = ts_colors[i]
    xs, ys = [], []

    for ds in DEPTHS_ORDER:
        d13c = resampled[ds].loc[ts, 'd13c']
        depth = DS_META[ds]['depth']
        if pd.notna(d13c):
            xs.append(d13c)
            ys.append(depth)
            # Plot point — diamond for F, circle for others
            mk = DS_META[ds]['marker']
            ec = DS_META[ds]['color']
            ax.plot(d13c, depth, marker=mk, ms=5, color=col,
                    markeredgecolor=ec, markeredgewidth=1.0, zorder=3, lw=0)

    # Connect available points with line
    if len(xs) > 1:
        # Sort by depth for clean line
        paired = sorted(zip(ys, xs))
        ys_s, xs_s = zip(*paired)
        ax.plot(xs_s, ys_s, color=col, lw=1.2, alpha=0.7, zorder=2)

# ── Depth horizon lines ───────────────────────────────────────────────────────
for ds in DEPTHS_ORDER:
    depth = DS_META[ds]['depth']
    col   = DS_META[ds]['color']
    ls    = '--' if ds == 'F' else ':'
    lw    = 1.0  if ds == 'F' else 0.5
    ax.axhline(depth, color=col, lw=lw, ls=ls, alpha=0.4, zorder=1)
    ax.text(ax.get_xlim()[0] if ax.get_xlim()[0] != 0 else -3.6,
            depth, f' {depth} m', va='center', ha='left',
            color=col, fontsize=7.5, fontweight='bold')

# ── Colourbar for time ─────────────────────────────────────────────────────────
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, N-1))
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.02, fraction=0.03)
cbar.ax.yaxis.set_tick_params(color='#d4cfc8')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='#d4cfc8', fontsize=8)
# Label ticks with actual dates
tick_positions = np.linspace(0, N-1, 6)
tick_labels    = [timestep_idx[int(p)].strftime('%b %Y') for p in tick_positions]
cbar.set_ticks(tick_positions)
cbar.set_ticklabels(tick_labels)
cbar.set_label('Time step', color='#d4cfc8', fontsize=9)
cbar.outline.set_edgecolor('#232b38')
cbar.ax.set_facecolor('#151922')

# ── Depth legend (right side, dataset symbols) ────────────────────────────────
legend_elements = []
for ds in DEPTHS_ORDER:
    m = DS_META[ds]
    legend_elements.append(
        Line2D([0], [0], marker=m['marker'], color='none',
               markerfacecolor='#555', markeredgecolor=m['color'],
               markeredgewidth=1.5, markersize=7, label=m['label'])
    )
leg = ax.legend(handles=legend_elements, loc='lower right',
                framealpha=0.15, edgecolor='#232b38',
                labelcolor='#d4cfc8', fontsize=8,
                facecolor='#151922')

# ── Styling ───────────────────────────────────────────────────────────────────
ax.set_xlabel('δ¹³C (‰)', color='#d4cfc8', fontsize=10)
ax.set_ylabel('Depth (m)', color='#d4cfc8', fontsize=10)
ax.set_title('MEMO — Depth vs δ¹³C Profile\nbi-monthly timesteps, age-shifted',
             color='#e8c97a', fontsize=11, pad=12)

ax.tick_params(colors='#7a8090', labelsize=8)
ax.spines[:].set_color('#232b38')
ax.grid(axis='x', color='#1c2230', lw=0.6, zorder=0)
ax.invert_yaxis()   # deeper at bottom

# Invert X if desired (more negative d13c to left is standard)
ax.set_xlim(sorted([
    min(resampled[ds]['d13c'].min() for ds in DEPTHS_ORDER) - 0.15,
    max(resampled[ds]['d13c'].max() for ds in DEPTHS_ORDER) + 0.15
]))

ax.set_ylim(-162, -105)

plt.tight_layout()
plt.savefig('depth_d13c_profile.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("Saved: depth_d13c_profile.png")